<a href="https://colab.research.google.com/github/hwangho-kim/Transformer_Fewshot_PdM/blob/main/%EB%B0%98%EB%8F%84%EC%B2%B4_FDC_%EC%9D%B4%EC%83%81_%ED%83%90%EC%A7%80_%EC%A0%84%EC%B2%B4_%ED%8C%8C%EC%9D%B4%ED%94%84%EB%9D%BC%EC%9D%B8_R05.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.optim as optim
import time

# matplotlib 설정 (스타일 지정, 한글 폰트 배제)
plt.rcParams['axes.unicode_minus'] = False
sns.set_style("whitegrid")

print("=====================================================")
print("=== Phase 1 & 2: Load Real CSV & Preprocessing    ===")
print("=====================================================")
import os

# 1. 특정 컬럼만 읽어서 데이터프레임 생성
use_cols = ['ts', 'step_id', 'org_wf_id', 'param_alias', 'param_value']
try:
    df_raw = pd.read_csv('sample.csv', usecols=use_cols)
    print(f"-> Successfully loaded 'sample.csv'. Total records: {len(df_raw):,}")
except FileNotFoundError:
    print("-> [에러] 'sample.csv' 파일을 찾을 수 없습니다. 경로를 확인해주세요.")
    raise

# 2. 파이프라인 내부 변수명으로 매핑 (코드 호환성 유지)
df_raw = df_raw.rename(columns={
    'ts': 'TIMESTAMP',
    'step_id': 'STEP_ID',
    'org_wf_id': 'WAFER_ID',
    'param_alias': 'PARA_NAME',
    'param_value': 'VALUE'
})

# 시간순 보장을 위한 정렬
df_raw = df_raw.sort_values(by=['WAFER_ID', 'TIMESTAMP'])

# 3. Wafer와 Parameter 자동 탐지 (시간순서대로 웨이퍼 정렬)
first_timestamps = df_raw.groupby('WAFER_ID')['TIMESTAMP'].min().sort_values()
detected_wafers = first_timestamps.index.tolist()
detected_params = list(df_raw['PARA_NAME'].unique())
num_params = len(detected_params)
num_wafers = len(detected_wafers)

print(f"-> Auto-detected Wafers: {num_wafers} ea")
print(f"-> Auto-detected Parameters: {num_params} ea")

# 4. Long to Wide Format (Pivot) & 스텝 전환 시점 기록
print("-> Converting Long format to Wide format & Handling missing values...")
df_raw['RELATIVE_TIME'] = df_raw.groupby(['WAFER_ID', 'PARA_NAME']).cumcount()

step_transitions = {}
for w_id in detected_wafers:
    w_df = df_raw[df_raw['WAFER_ID'] == w_id].drop_duplicates(subset=['STEP_ID'], keep='first')
    step_transitions[w_id] = w_df['RELATIVE_TIME'].tolist()

df_wide = df_raw.pivot(index=['WAFER_ID', 'RELATIVE_TIME'], columns='PARA_NAME', values='VALUE').reset_index()

# 5. 3D Tensor 구성 (길이가 다른 웨이퍼들을 Max Length로 패딩)
seq_len = df_wide['RELATIVE_TIME'].max() + 1
tensor_data = np.zeros((num_wafers, seq_len, num_params))

for w_idx, w_id in enumerate(detected_wafers):
    w_data = df_wide[df_wide['WAFER_ID'] == w_id].sort_values('RELATIVE_TIME')
    current_len = len(w_data)

    # ffill(Forward Fill)로 센서 결측치 보완 후, 남은 결측치는 0으로 처리
    w_vals = w_data[detected_params].fillna(method='ffill').fillna(0).values
    tensor_data[w_idx, :current_len, :] = w_vals

    # 패딩: 공정이 일찍 끝난 웨이퍼는 마지막 센서 값을 유지시킴
    if current_len < seq_len and current_len > 0:
        tensor_data[w_idx, current_len:, :] = w_vals[-1, :]

params_list = detected_params

# 6. 정규화 (전체 웨이퍼 중 앞의 50%를 정상 베이스라인 모델 학습에 사용)
train_split_idx = max(1, int(num_wafers * 0.5))
train_data = tensor_data[:train_split_idx]

mean_val = train_data.mean(axis=(0, 1))
std_val = train_data.std(axis=(0, 1))

print("\n[Training Data Statistics - Selected Sensors (Top 4)]")
for i in range(min(4, num_params)):
    print(f" - {params_list[i]:>15}: Mean = {mean_val[i]:6.1f}, Std = {std_val[i]:5.1f}")

def normalize(data):
    return (data - mean_val) / (std_val + 1e-7)

def denormalize(data):
    return data * (std_val + 1e-7) + mean_val

tensor_data_norm = normalize(tensor_data)
train_data_norm = normalize(train_data)


print("\n=====================================================")
print("=== Phase 3: High-Performance Model Building      ===")
print("=====================================================")
class TS_MAE_Large(nn.Module):
    def __init__(self, num_features, d_model=256, nhead=8, num_layers=4):
        super().__init__()
        self.input_proj = nn.Linear(num_features, d_model)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, batch_first=True, dim_feedforward=1024, dropout=0.1
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.output_proj = nn.Linear(d_model, num_features)

    def forward(self, x, mask_ratio=0.15):
        batch_size, seq_len, _ = x.shape
        if mask_ratio > 0.0 and self.training:
            mask = torch.rand(batch_size, seq_len).to(x.device) < mask_ratio
            x_masked = x.clone()
            x_masked[mask] = 0.0
        else:
            x_masked = x

        encoded = self.input_proj(x_masked)
        out = self.transformer(encoded)
        return self.output_proj(out)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = TS_MAE_Large(num_features=num_params).to(device)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"-> Model Architecture: Transformer MAE")
print(f"-> Trainable Parameters: {total_params:,}")

if torch.cuda.device_count() > 1:
    print(f"-> Utilizing {torch.cuda.device_count()} GPUs with DataParallel!")
    model = nn.DataParallel(model)
else:
    print(f"-> Running on: {device}")

optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.MSELoss()

train_tensor = torch.FloatTensor(train_data_norm).to(device)
epochs = 80
batch_size = 16

model.train()
print("\nStarting Pre-training on Golden Wafers (1~100)...")
loss_history = []

start_train_time = time.time()
for epoch in range(epochs):
    epoch_loss = 0
    indices = torch.randperm(len(train_tensor))
    train_tensor_shuffled = train_tensor[indices]

    for i in range(0, len(train_tensor), batch_size):
        batch = train_tensor_shuffled[i:i+batch_size]
        optimizer.zero_grad()
        recon = model(batch, mask_ratio=0.20)
        loss = criterion(recon, batch)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()

    avg_loss = epoch_loss / (len(train_tensor)/batch_size)
    loss_history.append(avg_loss)
    if (epoch+1) % 20 == 0:
        print(f" - Epoch [{epoch+1:>3}/{epochs}], MSE Loss: {avg_loss:.4f}")

print(f"-> Training Complete. ({time.time() - start_train_time:.2f} sec)")

# [Visualization] Training Loss Curve
plt.figure(figsize=(8, 4))
plt.plot(loss_history, color='navy', linewidth=2)
plt.title('Self-Supervised Pre-training Loss Curve', fontsize=14)
plt.xlabel('Epochs', fontsize=12)
plt.ylabel('Reconstruction MSE Loss', fontsize=12)
plt.grid(True, alpha=0.4)
plt.tight_layout()
plt.show()


print("\n=====================================================")
print("=== Phase 4: Unsupervised Inference & Scatter Plot ===")
print("=====================================================")
def calculate_anomaly_scores(model, data_tensor):
    model.eval()
    with torch.no_grad():
        x = torch.FloatTensor(data_tensor).to(device)
        recon = model(x, mask_ratio=0.0)
        error_map = (x - recon) ** 2
        wafer_scores = torch.mean(error_map, dim=(1, 2)).cpu().numpy()
    return wafer_scores, error_map.cpu().numpy(), recon.cpu().numpy()

all_data_tensor = tensor_data_norm
anomaly_scores, error_maps, recon_data_norm = calculate_anomaly_scores(model, all_data_tensor)

train_scores = anomaly_scores[:100]
threshold = np.mean(train_scores) + 3 * np.std(train_scores) # 통계적 3-Sigma 임계치 적용

print(f"-> Normal Data Mean Score: {np.mean(train_scores):.4f}")
print(f"-> Normal Data Std Score:  {np.std(train_scores):.4f}")
print(f"-> Set Anomaly Threshold (Mean + 3*Std): {threshold:.4f}")

wafer_indices = np.arange(1, num_wafers + 1)
is_anomaly = anomaly_scores > threshold

plt.figure(figsize=(14, 6))
plt.scatter(wafer_indices[~is_anomaly], anomaly_scores[~is_anomaly],
            color='dodgerblue', alpha=0.6, label='Predicted Normal')
plt.scatter(wafer_indices[is_anomaly], anomaly_scores[is_anomaly],
            color='red', marker='x', s=80, linewidth=2, label='Predicted Anomaly')

plt.axhline(y=threshold, color='black', linestyle='--', label=f'Detection Threshold ({threshold:.4f})')

plt.title('Unsupervised Equipment Health Monitoring: Wafer Anomaly Scatter', fontsize=16)
plt.xlabel('Wafer Processing Sequence', fontsize=12)
plt.ylabel('Anomaly Score (Reconstruction Error)', fontsize=12)
plt.legend(loc='upper left')
plt.grid(True, alpha=0.4)
plt.tight_layout()
plt.show()


print("\n=====================================================")
print("=== Phase 5: Auto-Triggered Root Cause Analysis   ===")
print("=====================================================")
def generate_business_report(target_idx, rank):
    w_id = detected_wafers[target_idx]
    score = anomaly_scores[target_idx]
    e_map = error_maps[target_idx]

    print(f"\n[{rank} Rank Anomaly] RCA Report Generated for {w_id} (Score: {score:.4f})")

    plt.figure(figsize=(10, 3))
    sns.heatmap(e_map.T, cmap='Reds', cbar_kws={'label': 'Error Magnitude'})
    plt.yticks(ticks=np.arange(num_params)+0.5, labels=params_list, rotation=0, fontsize=9)
    plt.title(f'Root Cause Analysis X-Ray (Target: {w_id} - Auto Detected)', fontsize=12)
    plt.xlabel('Process Time (sec) / Step Progression', fontsize=10)

    # 동적 스텝 전환선 그리기 (실제 데이터 반영)
    transitions = step_transitions.get(w_id, [])
    for t_idx in transitions:
        if t_idx > 0:
            plt.axvline(t_idx, color='blue', linestyle=':', alpha=0.5)

    plt.tight_layout()
    plt.show()

    actual_trace = denormalize(all_data_tensor[target_idx])
    recon_trace = denormalize(recon_data_norm[target_idx])

    # 동적 서브플롯 생성 (센서 개수에 따라 알아서 줄 수 조절)
    num_rows = (num_params + 1) // 2
    fig, axes = plt.subplots(num_rows, 2, figsize=(14, max(4, 2 * num_rows)), sharex=True)
    if num_rows > 1:
        axes = axes.flatten()
    else:
        axes = np.array(axes).flatten()

    for p_idx, param_name in enumerate(params_list):
        axes[p_idx].plot(actual_trace[:, p_idx], label='Actual Sensor Value', color='black', linewidth=1.2)
        axes[p_idx].plot(recon_trace[:, p_idx], label='AI Expected Value', color='red', linestyle='--', linewidth=1.2)

        diff = np.abs(actual_trace[:, p_idx] - recon_trace[:, p_idx])
        dynamic_threshold = np.std(actual_trace[:, p_idx]) * 0.5
        axes[p_idx].fill_between(range(seq_len), 0, 1, where=(diff > dynamic_threshold),
                             color='red', alpha=0.2, transform=axes[p_idx].get_xaxis_transform())

        axes[p_idx].set_title(param_name, fontsize=10, pad=3)
        if p_idx == 0:
            axes[p_idx].legend(loc='upper right', fontsize=8)

        for t_idx in transitions:
            if t_idx > 0:
                axes[p_idx].axvline(t_idx, color='gray', linestyle=':', alpha=0.3)

    # 빈 그래프 숨기기
    for i in range(num_params, len(axes)):
        fig.delaxes(axes[i])

    # 하단 그래프에만 x축 라벨 표시
    for i in range(max(0, num_params - 2), num_params):
        axes[i].set_xlabel('Process Time (sec)', fontsize=10)

    plt.suptitle(f'Detailed Sensor Analysis: Actual vs AI Expectation ({w_id})', fontsize=14, y=0.98)
    plt.tight_layout()
    plt.show()

detected_anomalies = np.where(is_anomaly)[0]
print(f"-> Total Anomalies Detected > Threshold: {len(detected_anomalies)}")

if len(detected_anomalies) > 0:
    sorted_anomalies = detected_anomalies[np.argsort(anomaly_scores[detected_anomalies])[::-1]]

    print(f"-> Generating RCA reports for ALL {len(sorted_anomalies)} severe anomalies in descending order...")

    for rank, target_idx in enumerate(sorted_anomalies):
        generate_business_report(target_idx, rank + 1)
else:
    print("-> No anomalies detected above the given threshold.")


print("\n=====================================================")
print("=== Phase 6: Future Trend Prediction (Prognostics) ===")
print("=====================================================")
recent_window = max(5, int(num_wafers * 0.5)) # 동적 윈도우 설정
recent_wafers = wafer_indices[-recent_window:]
recent_scores = anomaly_scores[-recent_window:]

smoothed_scores = pd.Series(recent_scores).rolling(window=max(1, int(recent_window*0.1)), min_periods=1).mean().values
poly_coeffs = np.polyfit(recent_wafers, smoothed_scores, 2)
poly_func = np.poly1d(poly_coeffs)

future_horizon = max(10, int(num_wafers * 0.25))
future_wafers = np.arange(num_wafers + 1, num_wafers + future_horizon + 1)
future_predictions = poly_func(future_wafers)

# 수식 문자열 생성 (e.g., y = 1.2e-04x^2 + -3.4e-02x + 1.5e+00)
eq_str = f"Trend Eq: $y = {poly_coeffs[0]:.2e}x^2 + {poly_coeffs[1]:.2e}x + {poly_coeffs[2]:.2e}$"
print(f"-> Extracted Polynomial Function: {eq_str}")

plt.figure(figsize=(14, 6))

plt.scatter(wafer_indices, anomaly_scores, color='lightgray', s=20, label='Actual Anomaly Scores')
plt.plot(recent_wafers, smoothed_scores, color='blue', linewidth=2, label='Smoothed Trend (Recent 100 Wafers)')
plt.plot(future_wafers, future_predictions, color='red', linestyle='--', linewidth=2.5, label='Predicted Future Trend')

plt.axhline(y=threshold, color='black', linestyle='--', label=f'Warning Threshold ({threshold:.2f})')
critical_threshold = threshold * 2.0
plt.axhline(y=critical_threshold, color='darkred', linestyle='-', label=f'Critical Failure Limit ({critical_threshold:.2f})')

# 그래프 내에 수식 텍스트 삽입
plt.text(0.02, 0.85, eq_str, transform=plt.gca().transAxes, fontsize=12,
         bbox=dict(facecolor='white', alpha=0.8, edgecolor='gray'))

plt.title('Equipment Prognostics: Future Anomaly Trend Prediction', fontsize=16)
plt.xlabel('Wafer Processing Sequence (Including Future)', fontsize=12)
plt.ylabel('Anomaly Score', fontsize=12)
plt.legend(loc='upper left')
plt.grid(True, alpha=0.4)
plt.tight_layout()
plt.show()

print("\n=== Pipeline Execution Completed Successfully ===")